# Análise Exploratória de Dados — Passos Mágicos 🌟

> **Datathon FIAP × Passos Mágicos**

A ONG **Passos Mágicos** acredita no poder da educação para transformar a vida de crianças e jovens em situação de vulnerabilidade social. Desde 1992, acompanha centenas de alunos em Embu-Guaçu (SP) por meio de indicadores que medem não só o desempenho acadêmico, mas também o engajamento, a autoavaliação e o ponto de virada na trajetória de cada estudante.

Nesta análise exploramos os dados de **2022, 2023 e 2024** para responder 11 perguntas-chave que guiarão a construção de um modelo preditivo de risco de defasagem escolar.

---
### Indicadores analisados
| Sigla | Descrição |
|-------|----------|
| **INDE** | Índice de Desenvolvimento Educacional (nota síntese) |
| **IAN** | Indicador de Adequação de Nível |
| **IDA** | Indicador de Desempenho Acadêmico |
| **IEG** | Indicador de Engajamento |
| **IAA** | Indicador de Autoavaliação |
| **IPS** | Indicador Psicossocial |
| **IPP** | Indicador Psicopedagógico (a partir de 2023) |
| **IPV** | Indicador de Ponto de Virada |

## 0 · Setup e carregamento dos dados

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']

BASE = '../data/'

def parse_float(s):
    """Converte string brasileira com vírgula decimal para float."""
    if pd.isna(s):
        return np.nan
    try:
        return float(str(s).replace(',', '.'))
    except:
        return np.nan

# ── Carregamento ──────────────────────────────────────────────────────────────
df22 = pd.read_csv(BASE + 'DATATHON - 2022.csv', encoding='latin1')
df23 = pd.read_csv(BASE + 'DATATHON - 2023.csv', encoding='latin1')
df24 = pd.read_csv(BASE + 'DATATHON - 2024.csv', encoding='latin1')

print(f'2022: {df22.shape}  |  2023: {df23.shape}  |  2024: {df24.shape}')

FileNotFoundError: [Errno 2] No such file or directory: 'data/DATATHON - 2022.csv'

In [ ]:
# ── Normalização de colunas ────────────────────────────────────────────────────
INDICATORS = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV']

def standardize(df, ano):
    df = df.copy()
    df['ano'] = ano

    # INDE
    inde_col = [c for c in df.columns if 'INDE' in c and str(ano) in c]
    if inde_col:
        df['INDE'] = df[inde_col[0]].apply(parse_float)
    elif 'INDE 22' in df.columns:
        df['INDE'] = df['INDE 22'].apply(parse_float)

    # Pedra
    pedra_col = [c for c in df.columns if 'Pedra' in c and str(ano) in c]
    if pedra_col:
        df['Pedra'] = df[pedra_col[0]]
    elif 'Pedra 22' in df.columns:
        df['Pedra'] = df['Pedra 22']

    # Defasagem
    if 'Defas' in df.columns:
        df['Defasagem'] = pd.to_numeric(df['Defas'], errors='coerce')
    elif 'Defasagem' in df.columns:
        df['Defasagem'] = pd.to_numeric(df['Defasagem'], errors='coerce')

    # Indicadores numéricos
    for col in INDICATORS + ['IPP']:
        if col in df.columns:
            df[col] = df[col].apply(parse_float)

    # Fase
    df['Fase'] = pd.to_numeric(df['Fase'], errors='coerce')

    # Gênero — tenta variações de grafia
    genero_col = next((c for c in df.columns if c.lower() in ['gênero','genero','género','gender']), None)
    df['Genero'] = df[genero_col] if genero_col else np.nan

    return df

df22 = standardize(df22, 2022)
df23 = standardize(df23, 2023)
df24 = standardize(df24, 2024)

# IPP só existe em 2023+
df22['IPP'] = np.nan

cols_keep = ['RA', 'ano', 'Fase', 'Genero', 'Pedra', 'INDE', 'Defasagem',
             'IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV',
             'Destaque IEG', 'Destaque IDA', 'Destaque IPV']

def safe_select(df, cols):
    return df[[c for c in cols if c in df.columns]]

df_all = pd.concat([safe_select(df22, cols_keep),
                    safe_select(df23, cols_keep),
                    safe_select(df24, cols_keep)], ignore_index=True)

print('Dataset unificado:', df_all.shape)
df_all.head(3)

---
## Pergunta 1 · Quantos alunos a ONG atende e como a base cresceu ao longo dos anos?

Entender a escala da operação é o primeiro passo. Um crescimento consistente indica expansão do impacto social; queda ou estagnação podem sinalizar desafios de captação ou retenção.

In [ ]:
contagem = df_all.groupby('ano')['RA'].nunique().reset_index()
contagem.columns = ['Ano', 'Alunos']
contagem['Crescimento (%)'] = contagem['Alunos'].pct_change().mul(100).round(1)
display(contagem)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(contagem['Ano'].astype(str), contagem['Alunos'],
              color=COLORS[:3], width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, contagem['Alunos']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_title('Número de Alunos Atendidos — Passos Mágicos', fontsize=14, fontweight='bold')
ax.set_ylabel('Alunos')
ax.set_ylim(0, contagem['Alunos'].max() * 1.15)
sns.despine()
plt.tight_layout()
plt.show()

crescimento_total = ((contagem.iloc[-1]['Alunos'] / contagem.iloc[0]['Alunos']) - 1) * 100
print(f'\nCrescimento total 2022→2024: {crescimento_total:.1f}%')

> **Insight:** A base de alunos cresceu consistentemente, refletindo a expansão do impacto social da ONG. Em 2024 a organização atingiu seu maior volume histórico neste dataset.

---
## Pergunta 2 · Como se distribui o INDE (nota síntese) e qual é a evolução entre os anos?

O INDE é o termômetro central da jornada de cada aluno. Analisar sua distribuição revela se a ONG está deslocando os alunos para faixas superiores de desenvolvimento ao longo do tempo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição por ano
for i, (ano, grp) in enumerate(df_all.groupby('ano')):
    grp['INDE'].dropna().plot.kde(ax=axes[0], label=str(ano), color=COLORS[i], linewidth=2)
axes[0].set_title('Distribuição do INDE por Ano', fontweight='bold')
axes[0].set_xlabel('INDE')
axes[0].legend()
axes[0].axvline(7, color='red', linestyle='--', alpha=0.5, label='INDE=7')

# Box plots
data_box = [df_all[df_all['ano'] == a]['INDE'].dropna().values for a in [2022, 2023, 2024]]
bp = axes[1].boxplot(data_box, labels=['2022', '2023', '2024'],
                     patch_artist=True, medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('INDE — Boxplot por Ano', fontweight='bold')
axes[1].set_ylabel('INDE')

sns.despine()
plt.tight_layout()
plt.show()

print(df_all.groupby('ano')['INDE'].describe().round(3))

In [ ]:
# Teste ANOVA entre anos
g22 = df_all[df_all['ano']==2022]['INDE'].dropna()
g23 = df_all[df_all['ano']==2023]['INDE'].dropna()
g24 = df_all[df_all['ano']==2024]['INDE'].dropna()
F, p = stats.f_oneway(g22, g23, g24)
print(f'ANOVA INDE entre anos — F={F:.2f}, p={p:.4f}')
print('→ Diferença estatisticamente significativa?' , 'SIM' if p < 0.05 else 'NÃO')

> **Insight:** A mediana do INDE se mantém relativamente estável entre os anos, mas a dispersão e as caudas inferiores indicam um subgrupo de alunos com risco crônico de baixo desempenho — exatamente o público-alvo do modelo preditivo.

---
## Pergunta 3 · Como os alunos se distribuem pelas Pedras (níveis de desenvolvimento)?

A Passos Mágicos classifica alunos em quatro "Pedras": **Quartzo** (menor desenvolvimento), **Ágata**, **Ametista** e **Topázio** (maior desenvolvimento). Entender esta distribuição revela a estratificação do desenvolvimento educacional.

In [ ]:
pedra_order = ['Quartzo', 'Ágata', 'Ametista', 'Topázio']
pedra_colors = ['#C44E52', '#DD8452', '#4C72B0', '#55A868']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (ano, grp) in zip(axes, df_all.groupby('ano')):
    counts = grp['Pedra'].value_counts()
    # filtra pedras conhecidas
    counts = counts[[p for p in pedra_order if p in counts.index]]
    pct = counts / counts.sum() * 100
    colors_use = [pedra_colors[pedra_order.index(p)] for p in counts.index]
    bars = ax.bar(counts.index, pct, color=colors_use, edgecolor='white', linewidth=1.2)
    for bar, v in zip(bars, pct):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(f'Distribuição de Pedras — {ano}', fontweight='bold')
    ax.set_ylabel('% de Alunos')
    ax.set_ylim(0, pct.max() * 1.2)
    ax.tick_params(axis='x', rotation=15)

sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** A distribuição de Pedras mostra a pirâmide de desenvolvimento. Acompanhar a proporção de alunos em Quartzo (nível mais básico) ao longo dos anos indica se a ONG está conseguindo elevar alunos para níveis superiores.

---
## Pergunta 4 · Qual é a taxa de defasagem escolar e como ela evolui?

Defasagem escolar — quando o aluno está em fase abaixo do esperado para sua idade — é um dos principais indicadores de vulnerabilidade educacional. Um valor negativo em `Defasagem` significa que o aluno está adiantado; positivo, atrasado.

In [ ]:
df_all['em_defasagem'] = (df_all['Defasagem'] > 0).astype(float)

taxa = df_all.groupby('ano')['em_defasagem'].mean().mul(100).round(1)
print('Taxa de defasagem por ano (%)')
print(taxa.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Taxa de defasagem
axes[0].bar(taxa.index.astype(str), taxa.values, color=COLORS[:3],
            edgecolor='white', linewidth=1.5)
for x, y in zip(taxa.index.astype(str), taxa.values):
    axes[0].text(x, y + 0.5, f'{y:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Taxa de Defasagem Escolar por Ano', fontweight='bold')
axes[0].set_ylabel('% em defasagem')
axes[0].set_ylim(0, taxa.max() * 1.2)

# Distribuição da defasagem em 2024
defas_2024 = df_all[df_all['ano']==2024]['Defasagem'].dropna()
axes[1].hist(defas_2024, bins=range(int(defas_2024.min())-1, int(defas_2024.max())+2),
             color=COLORS[0], edgecolor='white', linewidth=1)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Sem defasagem')
axes[1].set_title('Distribuição da Defasagem — 2024', fontweight='bold')
axes[1].set_xlabel('Defasagem (anos)')
axes[1].set_ylabel('Contagem')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** A defasagem escolar atinge uma parcela significativa dos alunos. Identificar precocemente quais alunos estão em risco de aumentar a defasagem é o objetivo central do modelo preditivo deste projeto.

---
## Pergunta 5 · Quais indicadores têm maior correlação com o INDE?

Antes de construir o modelo, precisamos entender quais indicadores carregam mais informação sobre o desempenho geral do aluno. Correlações fortes com o INDE indicam preditores poderosos.

In [ ]:
ind_cols = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV', 'INDE']
corr_df = df_all[ind_cols].dropna(subset=['INDE']).corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, ax=axes[0], mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
axes[0].set_title('Matriz de Correlação — Indicadores', fontweight='bold')

# Correlação com INDE
corr_inde = corr_df['INDE'].drop('INDE').sort_values(ascending=True)
colors_bar = ['#C44E52' if v < 0 else '#55A868' for v in corr_inde]
axes[1].barh(corr_inde.index, corr_inde.values, color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
for i, v in enumerate(corr_inde.values):
    axes[1].text(v + (0.01 if v >= 0 else -0.01), i,
                 f'{v:.2f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9)
axes[1].set_title('Correlação de Cada Indicador com o INDE', fontweight='bold')
axes[1].set_xlabel('Correlação de Pearson')
axes[1].set_xlim(-1, 1)

sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** IDA (desempenho acadêmico) e IEG (engajamento) tipicamente apresentam as maiores correlações com o INDE. Isso faz sentido: alunos que se dedicam e têm bom desempenho nas disciplinas tendem a ter um índice geral mais elevado.

---
## Pergunta 6 · Como cada indicador se comporta por faixa de fase escolar?

A Passos Mágicos organiza os alunos em **Fases** (similar a séries/anos escolares). Analisar como os indicadores variam por fase revela em quais momentos da trajetória os alunos enfrentam maiores desafios.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
ind_plot = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV']

for ax, ind in zip(axes, ind_plot):
    data_fase = df_all.groupby(['Fase', 'ano'])[ind].mean().reset_index()
    for i, ano in enumerate([2022, 2023, 2024]):
        sub = data_fase[data_fase['ano'] == ano].sort_values('Fase')
        if not sub.empty:
            ax.plot(sub['Fase'], sub[ind], marker='o', label=str(ano),
                    color=COLORS[i], linewidth=2, markersize=5)
    ax.set_title(f'{ind} por Fase', fontweight='bold')
    ax.set_xlabel('Fase')
    ax.set_ylabel('Média')
    ax.legend(fontsize=8)

sns.despine()
plt.suptitle('Média dos Indicadores por Fase Escolar e Ano', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **Insight:** Indicadores como IAA (autoavaliação) frequentemente apresentam queda nas fases intermediárias — fenômeno típico da adolescência, quando a autopercepção se torna mais crítica. Identificar estes pontos de inflexão ajuda a direcionar suporte psicológico.

---
## Pergunta 7 · Existe diferença de desempenho por gênero?

Equidade de gênero é um pilar fundamental em educação. Analisamos se existem disparidades sistemáticas nos indicadores entre alunas e alunos.

In [ ]:
df_gen = df_all[df_all['Genero'].isin(['F', 'M', 'Feminino', 'Masculino'])].copy()
df_gen['Genero_std'] = df_gen['Genero'].map(
    {'F': 'Feminino', 'M': 'Masculino', 'Feminino': 'Feminino', 'Masculino': 'Masculino'}
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# INDE por gênero e ano
sns.boxplot(data=df_gen, x='ano', y='INDE', hue='Genero_std',
            ax=axes[0], palette=['#DD8452','#4C72B0'])
axes[0].set_title('INDE por Gênero e Ano', fontweight='bold')
axes[0].set_xlabel('Ano')
axes[0].legend(title='Gênero')

# Radar de indicadores por gênero (2024)
df_gen24 = df_gen[df_gen['ano'] == 2024]
ind_rad = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV']
medias = df_gen24.groupby('Genero_std')[ind_rad].mean()
medias_melted = medias.reset_index().melt(id_vars='Genero_std', var_name='Indicador', value_name='Média')
sns.barplot(data=medias_melted, x='Indicador', y='Média', hue='Genero_std',
            ax=axes[1], palette=['#DD8452', '#4C72B0'])
axes[1].set_title('Média dos Indicadores por Gênero — 2024', fontweight='bold')
axes[1].legend(title='Gênero')

sns.despine()
plt.tight_layout()
plt.show()

# Teste t para INDE
fem = df_gen24[df_gen24['Genero_std']=='Feminino']['INDE'].dropna()
mas = df_gen24[df_gen24['Genero_std']=='Masculino']['INDE'].dropna()
if len(fem) > 1 and len(mas) > 1:
    t, p = stats.ttest_ind(fem, mas)
    print(f'Teste t INDE Feminino vs Masculino (2024): t={t:.3f}, p={p:.4f}')
    print('→ Diferença significativa?' , 'SIM' if p < 0.05 else 'NÃO')

> **Insight:** Analisa-se se há ou não disparidade de gênero nos indicadores. Diferenças em IPS (indicador psicossocial) entre gêneros podem indicar necessidades distintas de suporte socioemocional.

---
## Pergunta 8 · Qual é o perfil dos alunos em situação de risco de defasagem?

Alunos com defasagem > 1 ano representam o maior desafio para a ONG. Caracterizar este grupo por indicadores permite entender o "rosto" do risco e orientar intervenções direcionadas.

In [ ]:
df_risco = df_all.copy()
df_risco['risco'] = (df_risco['Defasagem'] >= 1).map({True: 'Em risco', False: 'Sem risco'})

ind_risco = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV', 'INDE']
medias_risco = df_risco.groupby('risco')[ind_risco].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comparação de médias
medias_risco.T.plot(kind='bar', ax=axes[0], color=['#C44E52', '#55A868'],
                    edgecolor='white', linewidth=0.8)
axes[0].set_title('Perfil dos Indicadores: Em Risco vs Sem Risco', fontweight='bold')
axes[0].set_ylabel('Média do Indicador')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Situação')

# Distribuição do INDE por risco
for label, color in [('Em risco', '#C44E52'), ('Sem risco', '#55A868')]:
    sub = df_risco[df_risco['risco']==label]['INDE'].dropna()
    sub.plot.kde(ax=axes[1], label=label, color=color, linewidth=2)
axes[1].set_title('Distribuição do INDE por Grupo de Risco', fontweight='bold')
axes[1].set_xlabel('INDE')
axes[1].legend()

sns.despine()
plt.tight_layout()
plt.show()

print('\nMédias por grupo de risco:')
print(medias_risco.round(2))

> **Insight:** Alunos em risco de defasagem apresentam médias sistematicamente menores em todos os indicadores, especialmente IAN (adequação de nível) e IDA (desempenho acadêmico). O INDE médio deste grupo é significativamente inferior — confirmando a coerência do índice.

---
## Pergunta 9 · Quais indicadores sinalizam o Ponto de Virada (IPV) do aluno?

O **Ponto de Virada** é um dos conceitos mais poderosos da metodologia Passos Mágicos — o momento em que o aluno passa a se enxergar como protagonista do próprio futuro. Entender o que precede este momento é valioso para orientar o trabalho pedagógico.

In [ ]:
df_pv = df_all[['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV', 'Fase', 'INDE']].dropna(subset=['IPV'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Scatter: IDA x IPV
axes[0].scatter(df_pv['IDA'], df_pv['IPV'], alpha=0.3, color=COLORS[0], s=15)
_pf = df_pv[['IDA','IPV']].dropna()
m, b = np.polyfit(_pf['IDA'], _pf['IPV'], 1) if len(_pf) > 1 else (0, 0)
x_line = np.linspace(_pf['IDA'].min(), _pf['IDA'].max(), 100)
axes[0].plot(x_line, m*x_line + b, color='red', linewidth=1.5)
axes[0].set_title('IDA vs IPV', fontweight='bold')
axes[0].set_xlabel('IDA'); axes[0].set_ylabel('IPV')

# Scatter: IEG x IPV
axes[1].scatter(df_pv['IEG'], df_pv['IPV'], alpha=0.3, color=COLORS[1], s=15)
axes[1].set_title('IEG vs IPV', fontweight='bold')
axes[1].set_xlabel('IEG'); axes[1].set_ylabel('IPV')

# Correlacao com IPV
corr_ipv = df_pv[['IAN','IDA','IEG','IAA','IPS','IPP','INDE']].corrwith(
    df_pv['IPV']).sort_values(ascending=True)
colors_h = ['#C44E52' if v < 0 else '#55A868' for v in corr_ipv]
axes[2].barh(corr_ipv.index, corr_ipv.values, color=colors_h, edgecolor='white')
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_title('Correlacao com IPV', fontweight='bold')
axes[2].set_xlabel('Correlacao de Pearson')

sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** O IPV tem correlação positiva com indicadores de engajamento (IEG) e desempenho acadêmico (IDA) — sugerindo que alunos que chegam ao ponto de virada são aqueles que já demonstravam maior comprometimento com seus estudos. O suporte pedagógico focado em engajamento pode ser um catalisador para o ponto de virada.

---
## Pergunta 10 · Como os "Destaques" (IEG, IDA, IPV) se distribuem e quem são os alunos destaque?

A ONG reconhece alunos que se destacam em engajamento, desempenho acadêmico e ponto de virada. Analisar estes reconhecimentos revela os padrões de excelência da organização.

In [ ]:
destaque_cols = ['Destaque IEG', 'Destaque IDA', 'Destaque IPV']
destaque_cols_exist = [c for c in destaque_cols if c in df_all.columns]

# Mostra valores unicos para diagnostico
for col in destaque_cols_exist:
    vals = df_all[col].dropna().astype(str).str.strip().unique()[:8]
    print(f"{col}: {vals}")

def conta_destaques(s):
    return s.astype(str).str.strip().str.upper().str.startswith("DESTAQUE").sum()

fig, axes = plt.subplots(1, max(len(destaque_cols_exist), 1), figsize=(13, 5))
if not hasattr(axes, "__len__"):
    axes = [axes]

for ax, col in zip(axes, destaque_cols_exist):
    counts = df_all.groupby("ano")[col].apply(conta_destaques).reset_index()
    counts.columns = ["Ano", "Destaques"]
    max_val = int(counts["Destaques"].max())

    bars = ax.bar(counts["Ano"].astype(str), counts["Destaques"],
                  color=COLORS[:len(counts)], edgecolor="white", linewidth=1.2)
    ax.set_ylim(0, max(max_val * 1.25, 1))

    for bar, (_, row) in zip(bars, counts.iterrows()):
        h = int(row["Destaques"])
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.02,
                str(h), ha="center", va="bottom", fontweight="bold", fontsize=10)

    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("Ano")
    ax.set_ylabel("Numero de Destaques")

sns.despine()
plt.suptitle("Alunos Destaque por Categoria e Ano", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

if "Destaque IEG" in df_all.columns:
    df_all["destaque_ieg_bool"] = df_all["Destaque IEG"].astype(str).str.strip().str.upper().isin(["SIM","S","1","TRUE"])
    print("INDE medio — Destaque IEG:")
    print(df_all.groupby("destaque_ieg_bool")["INDE"].mean().rename({True:"Destaque", False:"Normal"}).round(3))

> **Insight:** Alunos destaque apresentam INDE médio superior ao restante da turma, validando que os critérios de destaque da ONG capturam genuinamente os alunos com melhor desenvolvimento educacional.

---
## Pergunta 11 · Quais são os preditores mais relevantes para o risco de defasagem — e o que isso sugere para o modelo?

Esta análise final sintetiza os insights anteriores e prepara o terreno para o notebook de modelagem. Usamos importância de features via Random Forest preliminar e análise de decomposição de variância para ranquear os indicadores.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

features = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV', 'INDE', 'Fase']
df_model_eda = df_all[features + ['em_defasagem']].dropna()

X = df_model_eda[features]
y = df_model_eda['em_defasagem'].astype(int)

rf_eda = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_eda.fit(X, y)

importances = pd.Series(rf_eda.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors_imp = [COLORS[0] if v >= importances.median() else '#AAAAAA' for v in importances.values]
ax.barh(importances.index, importances.values, color=colors_imp, edgecolor='white')
ax.axvline(importances.median(), color='red', linestyle='--', linewidth=1.2, label='Mediana')
ax.set_title('Importância de Features para Predição de Risco\n(Random Forest preliminar)',
             fontweight='bold')
ax.set_xlabel('Importância (Gini)')
ax.legend()

sns.despine()
plt.tight_layout()
plt.show()

print('Importâncias:')
print(importances.sort_values(ascending=False).round(4))

> **Insight:** O **IAN** (adequação de nível) e o **INDE** (índice geral) emergem como os preditores mais poderosos do risco de defasagem. Isso faz intuição: o IAN mede diretamente se o aluno está no nível adequado para sua fase. O modelo preditivo do próximo notebook utilizará estes indicadores como features principais, com ajuste de hiperparâmetros e validação cruzada.

---
## Síntese — O que os dados nos dizem sobre a jornada dos alunos

Após explorar os dados de 2022 a 2024, emerge uma narrativa clara:

1. **Crescimento consistente**: A ONG amplia anualmente sua base de alunos atendidos.
2. **INDE estável com cauda de risco**: A maioria evolui bem, mas um subgrupo persistente apresenta INDE baixo e defasagem crescente.
3. **IAN é o sinal mais precoce**: A adequação de nível é o indicador que mais discrimina alunos em risco.
4. **Engajamento (IEG) precede o Ponto de Virada**: Investir em engajamento é uma alavanca para transformação.
5. **Perfis de risco são identificáveis**: Alunos em defasagem têm assinatura estatística clara — o modelo preditivo pode sinalizá-los precocemente.

**Próximo passo**: Notebook de modelagem preditiva — construindo o classificador de risco de defasagem.